**Load features**

In [1]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np

In [2]:
train_features = np.load("features\\train_data.npy")
test_features = np.load("features\\test_data.npy")

train_labels = np.load("features\\train_labels.npy")
test_labels = np.load("features\\test_labels.npy")

# Transform
encoder = OneHotEncoder(sparse_output=False)
train_labels = encoder.fit_transform(train_labels.reshape(-1, 1))
test_labels = encoder.transform(test_labels.reshape(-1, 1))

standardScaler = StandardScaler()
train_features = standardScaler.fit_transform(train_features)
test_features = standardScaler.transform(test_features)

print(train_features.shape)
print(train_labels.shape)
print(test_features.shape)
print(test_labels.shape)

(5760, 162)
(5760, 8)
(288, 162)
(288, 8)


**Upload processed spectrogram images**

In [3]:
# Load data
train_images       = np.load("features\\train_images.npy")
test_images        = np.load("features\\test_images.npy")
train_image_labels = np.load("features\\train_image_labels.npy")
test_image_labels  = np.load("features\\test_image_labels.npy")

# One-hot encode labels trước
encoder = OneHotEncoder(sparse_output=False)
train_image_labels_encoded = encoder.fit_transform(train_image_labels.reshape(-1, 1))
test_image_labels_encoded  = encoder.transform(test_image_labels.reshape(-1, 1))

print(train_images.shape)
print(test_images.shape)
print(train_image_labels_encoded.shape)
print(test_image_labels_encoded.shape)

(5760, 224, 224)
(288, 224, 224)
(5760, 8)
(288, 8)


**Neural network architecture and training:**

In [8]:
import torch
import torch.nn as nn


class CNN2D_CNN1D(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        self.cnn2d = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.05),
            nn.Conv2d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(64, 512, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.5),
            nn.Conv2d(512, 256, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3),
            nn.Flatten(),
        )

        self.cnn1d = nn.Sequential(
            nn.Conv1d(1, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.25),
            nn.Conv1d(128, 256, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.45),
            nn.Conv1d(256, 128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.5),
            nn.Flatten(),
        )

        # Tự tính size
        with torch.no_grad():
            cnn2d_out = self.cnn2d(torch.zeros(1, 1, 64, 64)).shape[1]  # 4096
            cnn1d_out = self.cnn1d(torch.zeros(1, 1, 162)).shape[1]  # 2560

        self.classifier = nn.Sequential(
            nn.Linear(cnn2d_out + cnn1d_out, 128),  # 6656 → 128
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, num_classes),
            nn.Softmax(dim=1),
        )

    def forward(self, img, feat):
        x1 = self.cnn2d(img)
        x2 = self.cnn1d(feat)
        x = torch.cat([x1, x2], dim=1)
        return self.classifier(x)


# ← Khởi tạo lại model mới hoàn toàn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN2D_CNN1D(num_classes=8).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [9]:
from torch.utils.data import DataLoader, TensorDataset

# Ảnh grayscale (n, 64, 64) → (n, 1, 64, 64)
x_trainI = train_images[:, None, :, :] / 255.0
x_testI  = test_images[:, None, :, :]  / 255.0

# Feature (n, 162) → (n, 1, 162)
x_trainT = train_features.reshape(-1, 1, 162)
x_testT  = test_features.reshape(-1, 1, 162)

X_train_img  = torch.tensor(x_trainI, dtype=torch.float32)
X_test_img   = torch.tensor(x_testI,  dtype=torch.float32)
X_train_feat = torch.tensor(x_trainT, dtype=torch.float32)
X_test_feat  = torch.tensor(x_testT,  dtype=torch.float32)
y_train      = torch.tensor(train_labels, dtype=torch.float32)
y_test       = torch.tensor(test_labels,  dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train_img, X_train_feat, y_train),
    batch_size=32, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(X_test_img, X_test_feat, y_test),
    batch_size=32
)

In [10]:
# Kiểm tra shape thực của cả 2 branch
dummy_img  = torch.zeros(1, 1, 64, 64).to(device)
dummy_feat = torch.zeros(1, 1, 162).to(device)

out_img  = model.cnn2d(dummy_img)
out_feat = model.cnn1d(dummy_feat)

print(f"CNN2D output: {out_img.shape}")   # → thực tế
print(f"CNN1D output: {out_feat.shape}")  # → thực tế
print(f"Tổng:         {out_img.shape[1] + out_feat.shape[1]}")

CNN2D output: torch.Size([1, 4096])
CNN1D output: torch.Size([1, 2560])
Tổng:         6656
